# Lab 19 — UCI multicenter feature engineering

Đánh giá feature engineering trên 920 hàng UCI bằng Leave-One-Center-Out (LOCO).

Các biến mới theo đặc tả: `chol_per_age`, `bps_per_age`, `hr_ratio` và `age_bin`.

Age bin dùng ranh giới cố định `[0, 39, 49, 59, 69, 120]` để không học bins từ test hospital. Đây là feature engineering nghiên cứu, chưa phải biomarker lâm sàng đã được xác nhận.

In [ ]:
!pip -q install lightgbm seaborn

import json
import shutil
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score,
    brier_score_loss, confusion_matrix, f1_score, precision_score,
    recall_score, roc_auc_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
THRESHOLD = 0.50
OUTPUT_DIR = Path('/content/uci_multicenter_feature_engineering_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURES = ['age','sex','cp','trestbps','chol','fbs','restecg','thalach',
            'exang','oldpeak','slope','ca','thal']
TARGET = 'target'
NUMERICAL_FEATURES = ['age','trestbps','chol','thalach','oldpeak']
CATEGORICAL_FEATURES = ['sex','cp','fbs','restecg','exang','slope','ca','thal']
RATIO_FEATURES = ['chol_per_age','bps_per_age','hr_ratio']
AGE_BIN_FEATURE = 'age_bin'
AGE_BINS = [0, 39, 49, 59, 69, 120]
AGE_LABELS = [0, 1, 2, 3, 4]
BASE_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease'
FILES = {'cleveland':'processed.cleveland.data', 'hungarian':'processed.hungarian.data',
         'switzerland':'processed.switzerland.data', 'va':'processed.va.data'}
COLUMNS = FEATURES + ['num']

def read_uci(site, filename):
    frame = pd.read_csv(f'{BASE_URL}/{filename}', names=COLUMNS, na_values=['?'],
                        skipinitialspace=True).apply(pd.to_numeric, errors='coerce')
    frame[TARGET] = (frame['num'] > 0).astype('int8')
    frame['site'] = site
    return frame[FEATURES + [TARGET, 'site']]

data = pd.concat([read_uci(site, filename) for site, filename in FILES.items()], ignore_index=True)
assert len(data) == 920, f'Expected 920 rows, got {len(data)}'
SITES = list(FILES.keys())
display(data.groupby('site')[TARGET].agg(['size','sum','mean']).round(4))

## 1. P1 và các cấu hình FE

`F0` là baseline P1 đã dùng ở Lab 17/18. `F1` thêm ba ratio, `F2` thêm age bin, `F3` dùng toàn bộ feature mới. Tất cả config chạy cùng Logistic Regression và LightGBM, không class weight và không SMOTE.

In [ ]:
CONFIGS = {
    'F0_P1_baseline': {'ratios': False, 'age_bin': False},
    'F1_P1_ratios': {'ratios': True, 'age_bin': False},
    'F2_P1_age_bin': {'ratios': False, 'age_bin': True},
    'F3_P1_all_features': {'ratios': True, 'age_bin': True},
}

def apply_p1(frame):
    out = frame.copy()
    for column in FEATURES:
        out[column] = pd.to_numeric(out[column], errors='coerce')
    for column in ['trestbps', 'chol']:
        out.loc[out[column] <= 0, column] = np.nan
    return out

def safe_ratio(numerator, denominator):
    denominator = denominator.where(denominator > 0, np.nan)
    return (numerator / denominator).replace([np.inf, -np.inf], np.nan)

def make_features(frame, config):
    out = apply_p1(frame)
    if CONFIGS[config]['ratios']:
        out['chol_per_age'] = safe_ratio(out['chol'], out['age'])
        out['bps_per_age'] = safe_ratio(out['trestbps'], out['age'])
        out['hr_ratio'] = safe_ratio(out['thalach'], out['age'])
    if CONFIGS[config]['age_bin']:
        out['age_bin'] = pd.cut(out['age'], bins=AGE_BINS, labels=AGE_LABELS,
                                include_lowest=True).astype('object')
    return out

def feature_groups(config):
    numeric = NUMERICAL_FEATURES.copy()
    categorical = CATEGORICAL_FEATURES.copy()
    if CONFIGS[config]['ratios']:
        numeric += RATIO_FEATURES
    if CONFIGS[config]['age_bin']:
        categorical += [AGE_BIN_FEATURE]
    return numeric, categorical

def feature_columns(config):
    numeric, categorical = feature_groups(config)
    return numeric + categorical

display(pd.DataFrame({config: pd.Series(feature_columns(config)) for config in CONFIGS}).T)

In [ ]:
def build_pipeline(config, model_name):
    numeric_features, categorical_features = feature_groups(config)
    numeric = Pipeline([('imputer', SimpleImputer(strategy='median', add_indicator=True)),
                       ('scaler', StandardScaler())])
    categorical = Pipeline([('imputer', SimpleImputer(strategy='most_frequent', add_indicator=True)),
                            ('encoder', OneHotEncoder(handle_unknown='ignore'))])
    preprocessor = ColumnTransformer([('numeric', numeric, numeric_features),
                                     ('categorical', categorical, categorical_features)])
    if model_name == 'Logistic Regression':
        estimator = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
    else:
        estimator = LGBMClassifier(n_estimators=250, learning_rate=0.03, num_leaves=15,
            min_child_samples=15, subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
            random_state=RANDOM_STATE, verbosity=-1)
    return Pipeline([('preprocessor', preprocessor), ('classifier', estimator)])

def safe_roc_auc(y_true, probability):
    return roc_auc_score(y_true, probability) if len(np.unique(y_true)) > 1 else np.nan

def fit_score(config, model_name, train_frame, test_frame):
    train_ready = make_features(train_frame, config)
    test_ready = make_features(test_frame, config)
    columns = feature_columns(config)
    model = build_pipeline(config, model_name)
    started = time.perf_counter()
    model.fit(train_ready[columns], train_ready[TARGET])
    fit_seconds = time.perf_counter() - started
    probability = model.predict_proba(test_ready[columns])[:, 1]
    prediction = (probability >= THRESHOLD).astype(int)
    y_true = test_ready[TARGET]
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {'accuracy': accuracy_score(y_true, prediction),
        'precision': precision_score(y_true, prediction, zero_division=0),
        'recall': recall_score(y_true, prediction, zero_division=0),
        'pr_auc': average_precision_score(y_true, probability),
        'specificity': tn / (tn + fp) if (tn + fp) else np.nan,
        'f1': f1_score(y_true, prediction, zero_division=0),
        'roc_auc': safe_roc_auc(y_true, probability),
        'brier': brier_score_loss(y_true, probability),
        'false_negatives': int(fn), 'fit_seconds': fit_seconds}

def audit_features(config, train_frame, test_frame, test_site):
    train_ready = make_features(train_frame, config)
    test_ready = make_features(test_frame, config)
    columns = feature_columns(config)
    return {'configuration': config, 'test_site': test_site,
        'train_rows': len(train_ready), 'test_rows': len(test_ready),
        'train_missing_rate': float(train_ready[columns].isna().mean().mean()),
        'test_missing_rate': float(test_ready[columns].isna().mean().mean()),
        'feature_count': len(columns), 'ratio_features': int(CONFIGS[config]['ratios']),
        'age_bin_feature': int(CONFIGS[config]['age_bin'])}

records = []
audits = []
for config in CONFIGS:
    for test_site in SITES:
        train_frame = data[data['site'] != test_site].reset_index(drop=True)
        test_frame = data[data['site'] == test_site].reset_index(drop=True)
        audits.append(audit_features(config, train_frame, test_frame, test_site))
        for model_name in ['Logistic Regression', 'LightGBM']:
            metrics = fit_score(config, model_name, train_frame, test_frame)
            records.append({'configuration': config, 'model': model_name,
                'test_site': test_site, 'test_rows': len(test_frame), **metrics})

results = pd.DataFrame(records)
audits = pd.DataFrame(audits)
display(results.round(4))
print('LOCO result rows:', len(results))

## 2. Tổng hợp FE và xuất kết quả

Mọi delta đều so với `F0_P1_baseline` của cùng model. Vì mỗi cấu hình có đúng 4 test hospitals, `false_negatives_total` có thể so sánh trực tiếp trong Lab 19.

In [ ]:
summary = results.groupby(['configuration', 'model']).agg(
    folds=('test_site', 'nunique'), roc_auc_mean=('roc_auc', 'mean'),
    roc_auc_std=('roc_auc', 'std'), roc_auc_worst=('roc_auc', 'min'),
    pr_auc_mean=('pr_auc', 'mean'), recall_mean=('recall', 'mean'),
    recall_std=('recall', 'std'), recall_worst=('recall', 'min'),
    specificity_mean=('specificity', 'mean'), f1_mean=('f1', 'mean'),
    brier_mean=('brier', 'mean'), false_negatives_mean_per_fold=('false_negatives', 'mean'),
    false_negatives_total=('false_negatives', 'sum'), fit_seconds_mean=('fit_seconds', 'mean')).reset_index()

baseline = summary[summary['configuration'] == 'F0_P1_baseline'].set_index('model')
delta = summary.copy()
for metric in ['roc_auc_mean', 'roc_auc_worst', 'pr_auc_mean', 'recall_mean',
                'recall_worst', 'brier_mean', 'false_negatives_total']:
    delta[f'delta_vs_F0_{metric}'] = delta.apply(
        lambda row: row[metric] - baseline.loc[row['model'], metric], axis=1)

display(summary.sort_values(['model', 'roc_auc_worst'], ascending=[True, False]).round(6))
display(delta.round(6))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=summary, x='configuration', y='roc_auc_mean', hue='model', ax=axes[0])
axes[0].set_title('Mean ROC-AUC by FE configuration'); axes[0].tick_params(axis='x', rotation=25)
sns.barplot(data=summary, x='configuration', y='recall_mean', hue='model', ax=axes[1])
axes[1].set_title('Mean recall by FE configuration'); axes[1].tick_params(axis='x', rotation=25)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'feature_engineering_summary.png', dpi=180, bbox_inches='tight'); plt.show()

results_path = OUTPUT_DIR / 'feature_engineering_loco_results.csv'
audit_path = OUTPUT_DIR / 'feature_engineering_fold_audit.csv'
summary_path = OUTPUT_DIR / 'feature_engineering_summary.csv'
delta_path = OUTPUT_DIR / 'feature_engineering_delta_vs_baseline.csv'
results.to_csv(results_path, index=False); audits.to_csv(audit_path, index=False)
summary.to_csv(summary_path, index=False); delta.to_csv(delta_path, index=False)
run_config = {'dataset_rows': 920, 'validation': 'Leave-One-Center-Out',
    'preprocessing': 'P1_sentinel_aware', 'threshold': THRESHOLD,
    'feature_configurations': CONFIGS, 'age_bins': AGE_BINS,
    'models': ['Logistic Regression', 'LightGBM'], 'smote': False,
    'test_data_policy': 'real held-out hospital only'}
(OUTPUT_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2), encoding='utf-8')
zip_path = shutil.make_archive('/content/uci_multicenter_feature_engineering_results', 'zip', OUTPUT_DIR)
print('Saved:', OUTPUT_DIR, zip_path)